# NEXWAVE AI — RF Spectrum Analyzer v0.1

**Maqsad:** sun’iy IQ signal yaratish, FFT/PSD spektrini chizish va Isolation Forest yordamida noodatiy o‘lchovlarni topish.

> Ushbu notebook o‘quv prototipi. Sun’iy ma’lumotlardan olingan anomaliya natijasi noqonuniy uzatish yoki radioxalaqit isboti emas. Haqiqiy monitoring uchun ruxsat etilgan o‘lchovlar, kalibrlangan sensorlar va mutaxassis tekshiruvi zarur.

## 1. Kutubxonalarni o‘rnatish

In [ ]:
!pip -q install numpy pandas matplotlib scikit-learn

## 2. Kerakli kutubxonalarni chaqirish

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix

## 3. Sun’iy IQ signal yaratish

In [ ]:
# IQ — kompleks (I + jQ) ko‘rinishidagi raqamli signal namunalaridir.
sample_rate_hz = 2_000_000
duration_s = 0.02
n = int(sample_rate_hz * duration_s)
t = np.arange(n) / sample_rate_hz
rng = np.random.default_rng(7)

noise_std = 0.12
iq = noise_std * (
    rng.standard_normal(n) + 1j * rng.standard_normal(n)
) / np.sqrt(2)

# Ikki sun’iy ton: +120 kHz va -260 kHz
iq += 1.0 * np.exp(1j * 2 * np.pi * 120_000 * t)
iq += 0.45 * np.exp(1j * 2 * np.pi * (-260_000) * t)

print("Namuna soni:", len(iq))
print("Namuna olish chastotasi:", sample_rate_hz, "Hz")

## 4. FFT va Power Spectral Density (PSD)

In [ ]:
window = np.hanning(len(iq))
spectrum = np.fft.fftshift(np.fft.fft(iq * window))
power = (np.abs(spectrum) ** 2) / np.sum(window ** 2)
psd_db = 10 * np.log10(power + 1e-15)
freq_hz = np.fft.fftshift(np.fft.fftfreq(len(iq), d=1/sample_rate_hz))

plt.figure(figsize=(12, 4))
plt.plot(freq_hz / 1e3, psd_db, linewidth=0.8)
plt.title("NEXWAVE AI — Simulated RF Spectrum")
plt.xlabel("Frequency offset (kHz)")
plt.ylabel("Relative PSD (dB)")
plt.grid(True, alpha=0.3)
plt.show()

## 5. Spektrdagi asosiy cho‘qqilarni ko‘rish

In [ ]:
# Eng yuqori PSD qiymatlariga mos chastotalar (oddiy demo)
peak_indices = np.argsort(psd_db)[-8:][::-1]
peaks = pd.DataFrame({
    "frequency_offset_khz": freq_hz[peak_indices] / 1e3,
    "relative_psd_db": psd_db[peak_indices]
})
display(peaks)

## 6. AI uchun sun’iy RF o‘lchovlar to‘plami

In [ ]:
rng = np.random.default_rng(21)
n_normal = 250
n_anomaly = 18

normal = pd.DataFrame({
    "center_frequency_mhz": rng.normal(900, 1.8, n_normal),
    "bandwidth_khz": np.clip(rng.normal(180, 25, n_normal), 60, None),
    "signal_power_dbm": rng.normal(-68, 4, n_normal),
    "noise_floor_dbm": rng.normal(-96, 2, n_normal),
    "spectral_occupancy_pct": np.clip(rng.normal(32, 7, n_normal), 0, 100),
    "signal_duration_ms": np.clip(rng.normal(110, 25, n_normal), 5, None),
    "demo_ground_truth": 0
})

unusual = pd.DataFrame({
    "center_frequency_mhz": rng.choice([875, 925, 960, 840], n_anomaly) + rng.normal(0, 0.3, n_anomaly),
    "bandwidth_khz": rng.normal(520, 90, n_anomaly),
    "signal_power_dbm": rng.normal(-43, 5, n_anomaly),
    "noise_floor_dbm": rng.normal(-91, 3, n_anomaly),
    "spectral_occupancy_pct": np.clip(rng.normal(78, 8, n_anomaly), 0, 100),
    "signal_duration_ms": rng.normal(420, 70, n_anomaly),
    "demo_ground_truth": 1
})

df = pd.concat([normal, unusual], ignore_index=True)
df = df.sample(frac=1, random_state=21).reset_index(drop=True)
display(df.head())
print("Jami yozuvlar:", len(df))

## 7. Isolation Forest modelini o‘qitish

In [ ]:
feature_columns = [
    "center_frequency_mhz",
    "bandwidth_khz",
    "signal_power_dbm",
    "noise_floor_dbm",
    "spectral_occupancy_pct",
    "signal_duration_ms"
]
X = df[feature_columns]

model = IsolationForest(
    n_estimators=200,
    contamination=0.07,
    random_state=42
)
model.fit(X)

# Isolation Forest: -1 = anomaly, +1 = normal
df["model_prediction"] = (model.predict(X) == -1).astype(int)
df["anomaly_score"] = -model.score_samples(X)

display(df.sort_values("anomaly_score", ascending=False).head(12))

## 8. Model natijasini baholash

In [ ]:
print("Confusion matrix (qator: haqiqiy, ustun: model):")
print(confusion_matrix(df["demo_ground_truth"], df["model_prediction"]))
print()
print(classification_report(
    df["demo_ground_truth"],
    df["model_prediction"],
    target_names=["Normal", "Demo anomaly"],
    zero_division=0
))

## 9. Natijalarni grafikda ko‘rsatish

Quyidagi grafikda signal quvvati va bandlik bo‘yicha model belgilagan noodatiy kuzatuvlar ajratiladi.

In [ ]:
plt.figure(figsize=(8, 5))
normal_mask = df["model_prediction"] == 0
anomaly_mask = df["model_prediction"] == 1

plt.scatter(
    df.loc[normal_mask, "signal_power_dbm"],
    df.loc[normal_mask, "spectral_occupancy_pct"],
    label="Model: normal",
    alpha=0.65
)
plt.scatter(
    df.loc[anomaly_mask, "signal_power_dbm"],
    df.loc[anomaly_mask, "spectral_occupancy_pct"],
    label="Model: anomaly",
    marker="x",
    s=70
)
plt.xlabel("Signal power (dBm)")
plt.ylabel("Spectral occupancy (%)")
plt.title("NEXWAVE AI — Anomaly Detection Demo")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## 10. Natijalarni CSV faylga saqlash

In [ ]:
output_file = "nexwave_rf_anomaly_results.csv"
df.to_csv(output_file, index=False)
print("Fayl saqlandi:", output_file)

# Colab'dan kompyuterga yuklab olish uchun:
from google.colab import files
files.download(output_file)

## Keyingi bosqich

- Simulyatsiya o‘rniga ruxsat etilgan IQ yozuvini import qilish.
- Sensor metadata va o‘lchov kalibrovkasini qo‘shish.
- Modelni alohida train/test ma’lumotlarida tekshirish.
- RF hodisalarini xaritada ko‘rsatish.
- Keyin FastAPI backend va dashboard yaratish.

**Eslatma:** `contamination=0.07` — demo uchun tanlangan taxminiy parametr. Real tizimda u ma’lumotlar va tekshirilgan ground-truth asosida tanlanadi.